# ⚡ 14 · KV Cache 的内存分层与页级调度（Paged / Tiered / Prefetched）

> AI-Infra 面试高频：**长上下文时代，KV Cache 早就不是一张草稿纸，而是一个需要页表、分层存储和预取调度的小型内存系统。**

- 一个 128K 上下文、batch=8 的请求，GQA 模型的 KV 就有 **数百 GB**——比权重还大，任何单卡都装不下；
- 于是工程上把它当成**虚拟内存**来管：固定大小的**页（page）**、**页表（page table）**、**HBM 槽位**、**DRAM/SSD 分级**、**缺页搬运**与**投机预取**；
- 本节把「KV 分页 + 分层 + 预测式预取」拆成可跑的模拟器：自己实现 `KVTierManager`，用三类真实访问模式（长解码 / 多轮对话 / Agent 工具调用）对比 LRU、页大小、容量、前缀复用、预取与压缩的收益边界；
- 最后一节对照 **2026-11 的六个前沿工作**（TripleKV / Twindex / Aggron / TideCache / AscMLA / MoBA），说明工业界正在把同一套抽象推到哪一步。


## 📋 本节要解决的问题

| # | 问题 | 直观答案 | 量化口径 |
|---|---|---|---|
| 1 | KV Cache 到底占多少显存？ | 随「上下文长度 × batch」线性膨胀，长上下文时超过权重 | $2 \cdot L \cdot n_{kv} \cdot d_h \cdot \text{bytes}$ / token |
| 2 | 为什么 GQA / MLA 能把 KV 砍下来？ | 少存几份 K/V（GQA 共享 KV 头），或只存低秩 latent（MLA） | GQA ≈ **320 KB**/token vs MLA ≈ **69 KB**/token（BF16，见 §2） |
| 3 | 装不下怎么办？ | 分页 + 分层：热页留 HBM，冷页下沉 DRAM/SSD，缺页再搬回来 | 页大小 $P$ tokens，搬运 = 页字节 ÷ PCIe 带宽 |
| 4 | 搬运一次多贵？ | 一页几百 MB 量级，PCIe 上就是毫秒级停顿 | 320 KB/token × 16 token = 5 MB ≈ 0.2 ms @25 GB/s |
| 5 | 换页会不会把命中率换没？ | 会：访问模式决定一切，扫描型访问能让 LRU 命中率归零 | 见 §5 实验 1~3 |
| 6 | 有没有办法不被动挨打？ | **预测式预取**：按上一轮/上一层访问预测下一批页提前搬 | TripleKV 就是把这件事做到 KV 上（§6） |
| 7 | 为什么 Agent 场景要重做 KV 管理？ | Agent 反复整段读上下文、长尾扫描，访问模式与聊天完全不同 | 见 §5 Agent 模式与 §6 的 Aggron |


## 1️⃣ 直觉与大白话

### 1.1 草稿纸 → 抽屉 → 书架

把推理时的 KV Cache 想成**草稿纸**：每生成一个 token，就要把之前所有 token 的 K/V 摊在桌面上重读一遍。

- 短上下文：一张纸够用，桌子（HBM）放得下，**不需要任何管理**；
- 长上下文：纸越堆越高，桌面（80 GB HBM）放不下 → 只能把旧纸**按叠归档进抽屉（CPU DRAM）**，更冷的进**书架（NVMe SSD）**；
- 要用时再抽出来 → 这就是**分页（paging）+ 分层（tiering）**；抽出/放回的动作就是**缺页搬运（page fault / swap）**；
- 如果能提前猜到「下一段要读哪几叠」，趁计算时先把它们拿到桌面 → 这就叫**投机预取（speculative prefetch）**。

关键洞察：**桌面大小固定时，性能不取决于你用多聪明的淘汰算法，而取决于访问模式是否可预测**。这也是为什么本节要单独模拟三种访问模式。

### 1.2 四个必须背下来的名词

| 名词 | 含义 | 对应到这里 |
|---|---|---|
| **page（页）** | 固定大小的 KV 存储单位，含 $P$ 个 token 的 K/V | 页大小 $P\in\{8,16,32,64\}$ |
| **page table（页表）** | 「逻辑页号 → 物理槽位」的映射 | `request → [page_id] → slot` |
| **slot（槽位）** | HBM 里预留的一块固定大小缓存区 | `hbm_pages` 个 slot |
| **working set（工作集）** | 一段时间内真正被反复访问的页集合 | 决定命中率上界 |

### 1.3 与「专家分页」的镜像关系

同一套页表抽象，两个不同的生命周期（后面 §6 与面试考点会反复用到）：

| 维度 | KV 页（本节） | 专家页（第 11 讲） |
|---|---|---|
| 分配粒度 | 按**请求**分配，随序列增长 | 按 `(层, 专家下标)` 分配，**跨请求共享** |
| 生命周期 | 请求结束即回收 | 随**热度**反复换入换出 |
| 写入 | 追加写（append-only） | 只读（推理阶段） |
| 淘汰动机 | 请求变长、显存不够 | 专家太多、显存不够 |
| 预取依据 | 上一轮/上一层 attention 访问 | 上一层 router 输出 / 热度 |

一句话：**KV 分页省的是「碎片 + 容量」，专家分页省的是「14 ms 级搬运停顿」——目标函数不同，抽象相同。**


## 2️⃣ KV 账本：GQA vs MLA 差多少

### 2.1 公式

**GQA / MHA 口径**（每 token、每层缓存 K 和 V 两份）：

$$
\text{KV}_{\text{GQA}} = 2 \cdot L \cdot n_{kv} \cdot d_h \cdot b
$$

其中 $L$ 层数、$n_{kv}$ KV 头数（GQA 里远小于 query 头数）、$d_h$ 每头维度、$b$ 每元素字节数。

**MLA 口径**（DeepSeek 系：不存完整 K/V，只存低秩 latent $c^{KV}$ + RoPE 部分 $k^R$）：

$$
\text{KV}_{\text{MLA}} = L \cdot (d_c + d_r) \cdot b
$$

$d_c$ 为 latent 维度（如 512），$d_r$ 为解耦 RoPE 维度（如 64）。

### 2.2 注意两个陷阱

1. **单位**：$b=2$ 是 BF16；FP8 是 1，INT4 是 0.5——压缩直接影响的是**搬运字节**，不只是容量；
2. **口径**：MLA 省的是「KV 缓存字节」，代价是把计算挪到推理时的 up-projection（算力换显存）；面试时如果说成「MLA 不占显存」就错了。

下面是现场算出来的账本（后续所有实验都用这些常数）。


In [ ]:
# ---- matplotlib 中文字体（防图内中文乱码）----
import matplotlib
import numpy as np
import matplotlib.pyplot as plt
matplotlib.rcParams['font.sans-serif'] = ['Microsoft YaHei', 'SimHei', 'Songti SC',
                                          'Arial Unicode MS', 'DejaVu Sans']
matplotlib.rcParams['axes.unicode_minus'] = False
# -------------------------------------------

KB = 1024.0


def kv_gqa_bytes(L=80, n_kv=8, d_h=128, b=2):
    '''GQA：2 · L · n_kv · d_h · bytes（每 token）'''
    return 2 * L * n_kv * d_h * b


def kv_mla_bytes(L=61, d_c=512, d_r=64, b=2):
    '''MLA：L · (d_c + d_r) · bytes（每 token，只存 latent + RoPE）'''
    return L * (d_c + d_r) * b


GQA = kv_gqa_bytes()
MLA = kv_mla_bytes()

print('GQA 口径（L=80, n_kv=8, d_h=128, BF16）')
print(f'  每 token KV = {GQA:,.0f} B = {GQA/KB:.1f} KiB = {GQA/1000:.1f} KB')
print('MLA 口径（L=61, d_c=512, d_r=64, BF16）')
print(f'  每 token KV = {MLA:,.0f} B = {MLA/KB:.1f} KiB = {MLA/1000:.1f} KB')
print(f'  比值 MLA/GQA = {MLA/GQA:.3f}')

# 断言：口径区间（允许实现差异，但不允许数量级错）
assert 315_000 <= GQA <= 340_000, GQA
assert 65_000 <= MLA <= 95_000, MLA
assert MLA / GQA < 0.5

# ---------- 上下文长度 × batch 的显存账本 ----------
print('\n上下文 × batch=8 的 KV 显存（GB）')
print(f'{"ctx":>8}{"GQA":>10}{"MLA":>10}{"GQA/batch1":>14}')
for ctx in (4096, 32768, 131072):
    g = GQA * ctx * 8 / 1e9
    m = MLA * ctx * 8 / 1e9
    print(f'{ctx:>8}{g:>10.1f}{m:>10.1f}{GQA*ctx/1e9:>14.1f}')
    if ctx == 32768:
        assert g > 80, '32K/batch8 的 GQA KV 必须超过单卡 80 GB'
        assert m < 20, '同条件下 MLA 应远小于 20 GB'

print('\n读法：128K × batch8，GQA 需要 %.0f GB，MLA 只要 %.0f GB —— 这就是长上下文能落地的根因之一。'
      % (GQA * 131072 * 8 / 1e9, MLA * 131072 * 8 / 1e9))


### 2.3 KV 总量随上下文长度的增长

把上表的数字画成曲线，最容易看出为什么"分层"是必然：**横轴是上下文，纵轴是 KV 显存（双对数）**。GQA 在 128K 时单卡已经越线，MLA 还能留在卡内——这条线就是"要不要做 KV 分层"的判据。


In [ ]:
# KV 总量随上下文长度的增长：GQA vs MLA，batch=1 / 8
ctxs = np.array([1024, 4096, 16384, 65536, 131072])

fig, ax = plt.subplots(figsize=(7.6, 4.0))
for name, bpt_, marker in (('GQA', GQA, 'o'), ('MLA', MLA, 's')):
    for bs, ls in ((1, ':'), (8, '-')):
        gb = bpt_ * ctxs * bs / 1e9
        ax.plot(ctxs / 1024, gb, ls + marker, label=f'{name} · batch={bs}')
ax.set_xscale('log', base=2)
ax.set_yscale('log')
ax.set_xlabel('上下文长度（K tokens，log）'); ax.set_ylabel('KV 显存（GB，log）')
ax.set_title('KV 增长是指数级的：128K 上下文时 GQA 已进 TB 区间')
ax.grid(alpha=0.3, which='both'); ax.legend(fontsize=8)

# 单卡 80 GB 的可行边界（batch=1）
ax.axhline(80, color='tab:red', linestyle='--', linewidth=1)
ax.text(1024, 90, '单卡 80 GB 上限', color='tab:red', fontsize=9)
plt.tight_layout(); plt.show()

# 断言：曲线单调递增；同样上下文下 MLA 至少省 2×
kv_gqa_128k = GQA * 131072 / 1e9
kv_mla_128k = MLA * 131072 / 1e9
print(f'128K · batch1：GQA {kv_gqa_128k:.1f} GB vs MLA {kv_mla_128k:.1f} GB')
assert kv_gqa_128k > 40 and kv_mla_128k < 10
assert kv_gqa_128k / kv_mla_128k > 2.0


### 2.4 为什么"买更大显存"解决不了问题

一条经常被忽略的产业事实：

| 指标 | 增长速度 | 直观感受 |
|---|---|---|
| 单卡 HBM 容量 | 约 **2 年一倍**（80 GB → 141 GB → 192 GB…） | 慢 |
| 主流上下文窗口 | 约 **1 年 8×**（4K → 32K → 256K → 1M） | 快 |
| KV 每 token 字节 | 靠 GQA / MLA / 量化**阶段式**下降 | 中等 |

三者放在一起：**上下文增长速度 > 显存增长速度**，所以"KV 装不下"是长期结构性矛盾，不是某一代 GPU 的问题。工程上只能三选一（通常是三选三）：**压得更狠（量化/稀疏）、放的层级更多（分层）、搬得更聪明（预取/调度）**。


## 3️⃣ 机制图

![KV Cache 分层与页级调度](images/14_kv_tier_paging.svg)

**读图要点（面试可以直接照着讲）：**

1. **左侧逻辑视图**：一个请求的上下文被切成 $P$ 个 token 一页的逻辑页，页表 `request → [page_id]`；
2. **中间 HBM 槽位池**：只有 `hbm_pages` 个物理槽位，页表里的页号映射到槽号（`-1` 表示不在 HBM）；
3. **右侧分级存储**：DRAM（容量大、带宽低）与 NVMe（更大更慢）；命中不了就沿层级往下找；
4. **底部两条箭头**：红色是**缺页搬运**（同步，直接把计算卡住 → stall），绿色虚线是**预测式预取**（异步，走另一个 stream，尽量和计算重叠）；
5. **图上的数字**都用本节常数：页大小 $P$ tokens、单页字节 $P \times 320\,\text{KB}$、PCIe 有效带宽 25 GB/s。

> 注意：真实系统里还有第三种箭头——**压缩/重算**（把页压成 FP8/INT4，或者干脆丢掉重算），这就是 §6 里 AscMLA / Aggron 在做的事。


## 4️⃣ 手写实现

### 4.1 `KVTierManager` 的 API 契约

| 接口 | 语义 | 面试追问点 |
|---|---|---|
| `access(req, positions)` | 访问某请求的若干 token 位置，触发命中/缺页 | 为什么按**页**而不是按 token 命中？（粒度/元数据开销） |
| `append_tokens(req, n)` | 序列增长，分配新页 | 追加写为什么能做「只写一次」？ |
| `prefetch(req, pages)` | 投机预取若干页 | 预取错了怎么办？（浪费字节指标） |
| `hit_rate()` | 命中率 = hits / (hits+misses) | 冷启动算不算？（必须区分冷启动与稳态） |
| `stall_us()` | 缺页造成的停顿（字节 ÷ 带宽） | 真实系统里还可以用 overlap 掩盖 |
| `bytes_moved()` / `waste_rate()` | 搬运总量 / 预取浪费比例 | 为什么「只看命中率」会骗人？ |

### 4.2 三个实现细节（写错就废）

1. **淘汰必须失效页表**：把页踢出槽位时，`page_slot` 里那一项要删掉，否则两个逻辑页映射到同一个槽位（第 11 讲里的 double-booking bug，一模一样）；
2. **LRU 用 `OrderedDict`**：`move_to_end` 是 O(1)，别用 list 做 `remove`（O(n) 会把模拟拖垮）;
3. **预取要单独记账**：预取进来的页如果没被用到就被淘汰 → 计入 `wasted`，否则「命中率很高」可能只是因为搬了一堆没用的东西。

下面先写管理器，再写访问模式生成器。


In [ ]:
from collections import OrderedDict


class KVTierManager:
    '''两层（HBM / DRAM）KV 页管理器：页表 + 槽位 + LRU + 投机预取。'''

    def __init__(self, page_tokens=16, hbm_pages=256, bytes_per_token=327_680,
                 pcie_gbs=25.0):
        self.P = page_tokens
        self.cap = hbm_pages
        self.bytes_per_token = bytes_per_token
        self.bytes_per_page = float(bytes_per_token * page_tokens)
        self.bw = pcie_gbs * 1e9            # B/s

        self.hbm = {}                       # slot -> page_id
        self.page_slot = {}                 # page_id -> slot（不在 HBM 则无此键）
        self.free = list(range(hbm_pages))  # 空闲槽位
        self.lru = OrderedDict()            # page_id 顺序 = 访问新旧

        self.hits = 0
        self.misses = 0
        self.real_bytes = 0.0               # 真实缺页搬运
        self.pf_bytes = 0.0                 # 预取搬运
        self.wasted_bytes = 0.0             # 预取了但没用就被淘汰
        self.prefetched = set()

    # ---------------- 内部：装入 / 淘汰 ----------------
    def _evict_one(self):
        pid, _ = self.lru.popitem(last=False)      # 最久未用
        slot = self.page_slot.pop(pid)
        del self.hbm[slot]
        # 注意：这里不能把 slot 塞回 self.free —— 调用方会立刻用它装新页，
        # 若同时留在 free 里，下一次分配就会 double-booking。
        if pid in self.prefetched:                 # 预取白搬了
            self.prefetched.discard(pid)
            self.wasted_bytes += self.bytes_per_page
        return slot

    def _load(self, pid, speculative=False):
        '''返回 True 表示本来就在 HBM；False 表示发生了搬运。'''
        if pid in self.page_slot:
            return True
        slot = self.free.pop() if self.free else self._evict_one()
        self.hbm[slot] = pid
        self.page_slot[pid] = slot
        self.lru[pid] = None
        if speculative:
            self.pf_bytes += self.bytes_per_page
            self.prefetched.add(pid)
        else:
            self.real_bytes += self.bytes_per_page
        return False

    # ---------------- 公开接口 ----------------
    def append_tokens(self, req, n_tokens, start_pos):
        '''序列增长：把 [start_pos, start_pos+n) 覆盖到的逻辑页登记进页表。'''
        lo, hi = start_pos // self.P, (start_pos + n_tokens - 1) // self.P
        pages = list(range(lo, hi + 1))
        return pages

    def access(self, req, positions):
        '''访问若干 token 位置 → 折算成页 → 逐页命中/缺页。'''
        pages = sorted({p // self.P for p in positions})
        for pi in pages:
            pid = (req, pi)
            hit = self._load(pid, speculative=False)
            if hit:
                self.hits += 1
                self.prefetched.discard(pid)       # 预取命中，计入有用
            else:
                self.misses += 1
                self.prefetched.discard(pid)
            self.lru.move_to_end(pid)

    def prefetch(self, req, page_idx_list):
        '''投机预取：只装入，不参与命中统计。'''
        for pi in page_idx_list:
            pid = (req, pi)
            if pid not in self.page_slot:
                self._load(pid, speculative=True)

    # ---------------- 指标 ----------------
    def hit_rate(self):
        tot = self.hits + self.misses
        return self.hits / tot if tot else 0.0

    def bytes_moved(self):
        return self.real_bytes + self.pf_bytes

    def stalled_us(self, overlap=0.9):
        '''真实缺页全部串行；预取只暴露 (1-overlap) 的残余停顿。'''
        return (self.real_bytes + (1.0 - overlap) * self.pf_bytes) / self.bw * 1e6

    def waste_rate(self):
        return self.wasted_bytes / self.pf_bytes if self.pf_bytes else 0.0

    def resident_pages(self):
        return len(self.page_slot)

    # 便于自检
    def sanity(self):
        assert self.resident_pages() <= self.cap
        assert len(self.page_slot) == len(self.hbm) == len(self.lru)
        return True


print('KVTierManager 定义完成：页表 / 槽位池 / LRU / 预取 / 六项指标')


In [ ]:
import math


def simulate_access(mode, ctx=4096, steps=128, window=512, chunk=512, sink=16,
                    turns=4, shared_frac=0.0, n_req=3, n_access=128, seed=0):
    '''返回 [(req, positions), ...]：一段可复现的 KV 访问轨迹。

    mode:
      decode        长解码：每步读「attention sink + 最近 window 个 token」
      dialogue      多轮对话：每轮整段重读前缀（前缀共享比例 shared_frac）
      agent_scan    Agent 工具调用：顺序整段扫读上下文（长上下文总结场景）
      agent_random  Agent 工具调用：窗口随机撒在全上下文（长尾工具结果回查）
    '''
    rng = np.random.default_rng(seed)
    seq = []

    if mode == 'decode':
        for t in range(ctx, ctx + steps):
            seq.append((0, list(range(0, sink)) + list(range(t - window, t))))

    elif mode == 'dialogue':
        for r in range(n_req):
            shared = int(shared_frac * ctx)
            for turn in range(turns):
                end = shared + int((ctx - shared) * (turn + 1) / turns)
                seq.append((r, list(range(0, end))))

    elif mode == 'agent_scan':
        n = math.ceil(ctx / chunk)
        for _ in range(turns):
            for c in range(n):
                seq.append((0, list(range(c * chunk, min((c + 1) * chunk, ctx)))))

    elif mode == 'agent_random':
        n = math.ceil(ctx / window)
        for _ in range(n_access):
            c = int(rng.integers(0, n))
            seq.append((0, list(range(c * window, min((c + 1) * window, ctx)))))
    else:
        raise ValueError(mode)

    return seq


def replay(seq, page_tokens, hbm_pages, bytes_per_token, prefetch_lookahead=0,
           overlap=0.9, share_tokens=0):
    '''把轨迹灌进管理器跑一遍。

    share_tokens > 0 时，所有请求在 [0, share_tokens) 的前缀共享同一批物理页
    （真实系统里叫 prefix caching）：这部分页用命名空间 'shared-prefix'。
    '''
    m = KVTierManager(page_tokens, hbm_pages, bytes_per_token)
    for req, pos in seq:
        prefix = [p for p in pos if p < share_tokens]
        own = [p for p in pos if p >= share_tokens]
        if prefetch_lookahead:                     # 预测：把「当前窗口之后的一段」先搬进来
            lo, hi = min(pos), max(pos)
            width = hi - lo + 1
            fut = list(range(hi, hi + width * prefetch_lookahead + 1))
            m.prefetch(req, sorted({p // m.P for p in fut}))
        if prefix:
            m.access('shared-prefix', prefix)
        if own:
            m.access(req, own)
    m.sanity()
    return m


print('访问模式生成器就绪：decode / dialogue / agent_scan / agent_random')


def summarize(m):
    return (f'命中率 {m.hit_rate():.3f} | 缺页 {m.misses:5d} | 搬运 {m.bytes_moved()/1e6:7.1f} MB '
            f'| 停顿 {m.stalled_us()/1e3:8.2f} ms | 预取浪费 {m.waste_rate():.1%}')


# sanity：同一轨迹两次运行必须完全一致（可复现性）
s1 = replay(simulate_access('decode', ctx=1024, steps=16), 16, 64, GQA)
s2 = replay(simulate_access('decode', ctx=1024, steps=16), 16, 64, GQA)
assert (s1.hits, s1.misses, s1.bytes_moved()) == (s2.hits, s2.misses, s2.bytes_moved())
print('可复现性检查通过：', summarize(s1))


### 4.3 真实引擎是怎么实现这张页表的（vLLM / SGLang 对照）

| 机制 | vLLM 的叫法 | 本节对应实现 | 差异点 |
|---|---|---|---|
| 固定大小页 | `block_size = 16`（token 数） | `page_tokens = P` | 生产里基本固定 16，不随请求变 |
| 逻辑→物理页表 | `block_table[seq_id] -> [block_id]` | `page_slot[(req, pi)] -> slot` | vLLM 用 GPU 上的 int 数组，供 attention kernel 直接索引 |
| 物理页池 | `free_block_queue` | `free` 栈 + `hbm` | 生产里用双向链表/队列，O(1) 分配回收 |
| 引用计数共享 | `ref_cnt`（并行采样 / beam search / 前缀缓存共用） | 下一格的 COW 演示 | 同一前缀被多个序列共享时不复制 |
| 回收 | 完成即归还 + 前缀缓存 LRU | `_evict_one` | 生产里区分「已释放」与「可被前缀缓存复用」两类空闲 |

**三个实现层面的坑（面试加分项）：**

1. **页表要能被 kernel 直接读**：vLLM 把 `block_table` 放在显存里，attention kernel 按页索引 K/V；如果页表在 CPU 上，每一步都要拷贝，直接吃掉收益；
2. **写时复制（COW）是并行采样的必需品**：同一 prompt 采样 4 条回复，前 90% 的页完全一样，复制 4 份 KV 是纯浪费；COW 只在真正写入时才分裂；
3. **碎片在「页内」而不是「页间」**：最后一页平均浪费 $P/2$ 个 token 的空间——这就是为什么 $P$ 不能太大（E1 的另一面）。


In [ ]:
class PrefixCache:
    '''前缀缓存的最小实现：引用计数 + 写时复制（COW）。'''

    def __init__(self):
        self.ref = {}          # page_id -> 引用计数
        self.table = {}        # req -> [page_id]
        self.next_pid = 0
        self.copies = 0

    def _new_page(self):
        pid = self.next_pid
        self.next_pid += 1
        self.ref[pid] = 1
        return pid

    def build(self, req, n_pages):
        self.table[req] = [self._new_page() for _ in range(n_pages)]

    def fork(self, parent, child):
        '''共享父请求的全部页：只加引用计数，不复制数据。'''
        for pid in self.table[parent]:
            self.ref[pid] += 1
        self.table[child] = list(self.table[parent])

    def write(self, req, page_idx):
        '''写时复制：引用计数 > 1 才真正分裂出新页。'''
        pid = self.table[req][page_idx]
        if self.ref[pid] > 1:
            self.ref[pid] -= 1
            self.table[req][page_idx] = self._new_page()
            self.copies += 1
            return True
        return False

    def physical_pages(self):
        return len(self.ref)


pc = PrefixCache()
pc.build('r0', 64)                       # 母请求：64 页前缀
for r in ('r1', 'r2', 'r3'):
    pc.fork('r0', r)                     # 3 个并行采样共享前缀

pages_after_fork = pc.physical_pages()
print(f'fork 后物理页 {pages_after_fork}（朴素复制需要 {64*4}）')
assert pages_after_fork == 64            # 共享：一页都不多

for r in ('r1', 'r2', 'r3'):
    for pi in (10, 30):                  # 每条采样各写 2 页
        pc.write(r, pi)

pages_after_write = pc.physical_pages()
print(f'写后物理页 {pages_after_write}，发生 COW {pc.copies} 次')
assert pages_after_write == 64 + pc.copies == 70
assert pc.copies == 6
print('结论：4 条并行采样只多花 %.1f%% 的 KV 页（(70-64)/64），而不是 300%%。'
      % ((pages_after_write - 64) / 64 * 100))


## 5️⃣ 数值实验

五个实验，对应五个工程问题：

| 实验 | 问题 | 变量 | 看什么 |
|---|---|---|---|
| **E1** | 页多大合适？ | $P\in\{8,16,32,64\}$ × HBM 容量 | 命中率 / 搬运字节 / 停顿 |
| **E2** | 上下文变长，命中率什么时候崩？ | ctx $\in\{4K,16K,64K\}$ | 命中率崩塌曲线 |
| **E3** | 前缀缓存值多少钱？ | 前缀共享比例 $\in\{0,25,50,75\%\}$ | 命中率与停顿 |
| **E4** | 预测式预取能拿回多少？ | 无预取 vs 前瞻 1 段 vs 2 段 | 停顿、加速比、浪费率 |
| **E5** | 压缩（FP8/INT4）到底省什么？ | 固定 256 MB 预算 | 等效容量、搬运字节 |

**统一口径**（很重要，面试官会追问）：

- 页字节 = `bytes_per_token × P`，缺失一次就搬一整页；
- 命中率按**页访问次数**统计，不含预取；
- 停顿 = 真实缺页字节 ÷ 25 GB/s（PCIe 4.0 有效带宽）+ 预取残余 `(1-overlap)`；
- 所有随机轨迹固定 seed，结论可复现。


In [ ]:
seq = simulate_access('decode', ctx=4096, steps=128, window=512)
Ps = [8, 16, 32, 64]
caps = [512, 768, 1024]

print('E1：页大小 × HBM 容量（decode 轨迹，ctx=4K，window=512）')
print(f'{"cap( tokens)":>13} | ' + ' | '.join(f'P={p:<3}' for p in Ps))
table = {}
for cap_tokens in caps:
    hits, moved = [], []
    for P in Ps:
        m = replay(seq, P, max(4, cap_tokens // P), GQA)
        table[(cap_tokens, P)] = m
        hits.append(m.hit_rate())
        moved.append(m.bytes_moved() / 1e6)
    print(f'{cap_tokens:>13} | ' + ' | '.join(f'{h:.3f}' for h in hits))
    print(f'{"搬运 MB":>13} | ' + ' | '.join(f'{mv:7.1f}' for mv in moved))

# 断言 1：同一容量下，页越大搬运字节越多（过度取数）
for cap_tokens in caps:
    mv = [table[(cap_tokens, P)].bytes_moved() for P in Ps]
    assert all(b >= a * 0.999 for a, b in zip(mv, mv[1:])), (cap_tokens, mv)

# 断言 2：命中率随容量单调不减
for P in Ps:
    hs = [table[(c, P)].hit_rate() for c in caps]
    assert all(b >= a - 1e-9 for a, b in zip(hs, hs[1:])), (P, hs)

fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
for cap_tokens in caps:
    axes[0].plot(Ps, [table[(cap_tokens, P)].hit_rate() for P in Ps], 'o-', label=f'HBM={cap_tokens} tokens')
    axes[1].plot(Ps, [table[(cap_tokens, P)].bytes_moved() / 1e6 for P in Ps], 's--', label=f'HBM={cap_tokens} tokens')
axes[0].set_xlabel('页大小 P（tokens/page）'); axes[0].set_ylabel('命中率'); axes[0].set_title('命中率 vs 页大小')
axes[1].set_xlabel('页大小 P（tokens/page）'); axes[1].set_ylabel('搬运字节（MB）'); axes[1].set_title('搬运量 vs 页大小')
for ax in axes:
    ax.grid(alpha=0.3); ax.legend(fontsize=8)
plt.tight_layout(); plt.show()


**E1 解读（四条可以直接背的结论）：**

1. **容量差一页，命中率断崖**：HBM 只留 512 tokens（32 页）时，工作集（sink + 最近 512 tokens ≈ 33 页）正好装不下 → 命中率直接归零；把容量提到 768 tokens（跨过工作集）后命中率跳到 ~99%。**这不是「页大小」的问题，而是「容量 vs 工作集」的相变**——真实系统里表现为「batch 再加一个请求，P99 突然翻倍」；
2. **同样容量下，页越大搬运字节越多**：512 tokens 容量时 $P=8$ 搬 22.4 GB、$P=64$ 搬 26.8 GB；1024 tokens 时 215 MB vs 231 MB——一次缺页搬整页，页尾经常「还没用到就先搬了」；
3. **页越小，缺页次数越多**：本例 $P=8$ 的缺页次数约是 $P=64$ 的 8 倍，真实系统里每次缺页都有固定的调度/中断开销（本节的纯带宽模型没有计入这部分）；
4. **工程甜点区通常是 16~64 tokens/页**（vLLM 默认 block = 16）：既能摊薄元数据，又不会过度取数。**页大小不是越大越好，也不是越小越好，取决于访问局部性与带宽的比值。**


In [ ]:
print('E2：上下文变长 → 命中率崩塌（agent_random 轨迹，窗口 512，固定 128 次访问）')
ctxs = [4096, 16384, 65536]
cap_tokens = 4096
res2 = []
for ctx in ctxs:
    seq = simulate_access('agent_random', ctx=ctx, window=512, n_access=128, seed=3)
    m = replay(seq, 16, cap_tokens // 16, GQA)
    res2.append(m)
    print(f'  ctx={ctx:>6} : {summarize(m)}')

h2 = [m.hit_rate() for m in res2]
st2 = [m.stalled_us() / 1e3 for m in res2]
# 断言：命中率随上下文严格下降，停顿严格上升
assert h2[0] > h2[1] > h2[2], h2
assert st2[0] < st2[1] < st2[2], st2
assert h2[0] > 0.85 and h2[2] < 0.30, h2

fig, ax1 = plt.subplots(figsize=(7.2, 3.8))
ax1.plot([c // 1024 for c in ctxs], [h * 100 for h in h2], 'o-', color='tab:blue', label='命中率')
ax1.set_xlabel('上下文长度（K tokens）'); ax1.set_ylabel('命中率（%）', color='tab:blue')
ax2 = ax1.twinx()
ax2.plot([c // 1024 for c in ctxs], st2, 's--', color='tab:red', label='停顿')
ax2.set_ylabel('缺页停顿（ms）', color='tab:red')
ax1.set_title('HBM 容量固定 4096 tokens：上下文越长，命中率越低')
ax1.grid(alpha=0.3)
plt.tight_layout(); plt.show()


In [ ]:
print('E3：前缀复用比例 → 命中率（dialogue 轨迹，ctx=4K，4 轮，3 个请求）')
fracs = [0.0, 0.25, 0.5, 0.75]
res3 = []


def throughput_proxy(m):
    '''吞吐代理：总 KV 页访问次数 / 停顿（越大越好）。'''
    total_pages = m.hits + m.misses
    return total_pages / (m.stalled_us() / 1e6 + 1e-9)


for frac in fracs:
    seq = simulate_access('dialogue', ctx=4096, turns=4, shared_frac=frac, n_req=3)
    m = replay(seq, 16, 4096 // 16, GQA, share_tokens=int(frac * 4096))
    res3.append(m)
    print(f'  前缀共享 {frac:>4.0%} : {summarize(m)} | 吞吐代理 {throughput_proxy(m)/1e3:8.1f} K页/ms')

h3 = [m.hit_rate() for m in res3]
t3 = [throughput_proxy(m) for m in res3]
assert all(b >= a - 1e-9 for a, b in zip(h3, h3[1:])), h3
assert h3[-1] - h3[0] > 0.08, h3          # 75% 前缀共享至少要带来 8 个点
assert all(b >= a - 1e-9 for a, b in zip(t3, t3[1:])), t3

fig, ax = plt.subplots(figsize=(7.2, 3.8))
ax.bar([f'{int(f*100)}%' for f in fracs], [h * 100 for h in h3], color='tab:green', alpha=0.75)
for i, h in enumerate(h3):
    ax.text(i, h * 100 + 0.5, f'{h:.1%}', ha='center', fontsize=9)
ax.set_xlabel('请求间共享前缀比例'); ax.set_ylabel('命中率（%）')
ax.set_title('前缀缓存（prefix caching）：共享越多，KV 越不用重搬')
ax.grid(alpha=0.3, axis='y'); ax.set_ylim(0, 100)
plt.tight_layout(); plt.show()


**E2 / E3 解读：**

- **E2 是「容量墙」**：HBM 只留得住 4096 tokens 时，上下文从 4K 涨到 64K，命中率从 ~94% 掉到 ~21%（再涨到 64K 只剩 ~7%）——工作集（$ctx$）与容量之比决定了命中率上界，这条曲线在任何分层 KV 系统里都存在；
- **E3 是「复用墙」反向的一面**：共享前缀让命中率从 60.0% 提到 86.2%（+26.2 个点），停顿从 ~161 ms 降到 ~81 ms，吞吐代理从 ~11.9 涨到 ~34.6 K页/ms——**同一个请求集，只是把前缀改成共享物理页，三个指标同时变好**；生产环境里 prefix caching 之所以是第一优先级优化，就是因为**多个请求共享同一段 system prompt / 文档**是自然的复用；
- 但要小心「命中率高 = 快」的陷阱依然存在：如果共享前缀本身放不进容量（E2 的情形），命中率就会归零。**面试时务必按「命中率 → 搬运字节 → 停顿 → 吞吐」四层递进回答**，而不是只报第一个数。


In [ ]:
print('E4：预测式预取（agent_scan 轨迹，ctx=8K，chunk=512，容量 4096 tokens）')
seq = simulate_access('agent_scan', ctx=8192, chunk=512, turns=2)
variants = [
    ('无预取', 0),
    ('前瞻 1 段', 1),
    ('前瞻 2 段', 2),
]
res4 = []
for name, la in variants:
    m = replay(seq, 16, 4096 // 16, GQA, prefetch_lookahead=la)
    res4.append((name, m))
    print(f'  {name:<8}: {summarize(m)}')

base = res4[0][1]
for name, m in res4[1:]:
    print(f'  {name} 相对无预取：停顿 {base.stalled_us()/m.stalled_us():.2f}× 加速，'
          f'浪费率 {m.waste_rate():.1%}')

# 断言：预取必须显著降停顿；但必须承认浪费
assert res4[1][1].stalled_us() < base.stalled_us() * 0.5, '前瞻 1 段应至少砍掉一半停顿'
assert base.stalled_us() / res4[1][1].stalled_us() > 2.0
assert res4[1][1].waste_rate() < 0.15, res4[1][1].waste_rate()
# 更激进 ≠ 更好：前瞻 2 段的浪费率必须高于前瞻 1 段
assert res4[2][1].waste_rate() > res4[1][1].waste_rate() > 0.0

names = [n for n, _ in res4]
stalls = [m.stalled_us() / 1e3 for _, m in res4]
waste = [m.waste_rate() * 100 for _, m in res4]

fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
bars = axes[0].bar(names, stalls, color=['tab:gray', 'tab:orange', 'tab:red'], alpha=0.8)
for b, s in zip(bars, stalls):
    axes[0].text(b.get_x() + b.get_width() / 2, s, f'{s:.0f} ms', ha='center', va='bottom', fontsize=9)
axes[0].set_ylabel('缺页停顿（ms）'); axes[0].set_title('预取：把同步搬运变成重叠（停顿时长 ↓）')
axes[0].grid(alpha=0.3, axis='y')
axes[1].bar(names, waste, color=['tab:gray', 'tab:orange', 'tab:red'], alpha=0.8)
axes[1].set_ylabel('预取浪费率（%）'); axes[1].set_title('代价：预测错了就是白搬（浪费率 ↑）')
axes[1].grid(alpha=0.3, axis='y')
plt.tight_layout(); plt.show()


In [ ]:
print('E5：压缩（BF16 / FP8 / INT4）—— 固定 256 MB KV 预算下的等效容量与搬运量')
seq = simulate_access('agent_random', ctx=32768, window=512, n_access=128, seed=3)
budget = 256e6
res5 = []
for name, mult in (('BF16', 1), ('FP8', 2), ('INT4', 4)):
    bpt = GQA // mult
    pages = int(budget // (bpt * 16))
    m = replay(seq, 16, pages, bpt)
    res5.append((name, bpt, pages, m))
    print(f'  {name:<5}({bpt//1024:>4} KB/token): 可容纳 {pages:>4} 页 = {pages*16:>6} tokens | {summarize(m)}')

st5 = [m.stalled_us() / 1e3 for _, _, _, m in res5]
h5 = [m.hit_rate() for _, _, _, m in res5]
assert st5[0] > st5[1] > st5[2], st5           # 压缩越狠，搬运越少
assert st5[0] / st5[2] > 2.5, st5              # INT4 至少省 2.5× 停顿
assert all(b >= a - 1e-9 for a, b in zip(h5, h5[1:])), h5   # 容量变大 → 命中率不降

labels = [f'{n}\n{b//1024} KB/tok' for n, b, _, _ in res5]
fig, ax1 = plt.subplots(figsize=(7.6, 3.8))
bars = ax1.bar(labels, st5, color=['#4C78A8', '#72B7B2', '#54A24B'], alpha=0.85)
for b, s, h in zip(bars, st5, h5):
    ax1.text(b.get_x() + b.get_width() / 2, s, f'{s:.0f} ms\n命中 {h:.1%}', ha='center', va='bottom', fontsize=9)
ax1.set_ylabel('缺页停顿（ms）'); ax1.set_title('KV 压缩：同样的 256 MB 预算，能装下的 token 更多、搬得更少')
ax1.grid(alpha=0.3, axis='y')
plt.tight_layout(); plt.show()


**E4 / E5 解读：**

- **E4 预取的收益与代价同时出现**：前瞻 1 段把停顿从 ~215 ms 压到 ~35 ms（约 6×），代价是 ~3% 的浪费；前瞻 2 段浪费率翻到 ~6%，**停顿却没再降**（本例容量已足够覆盖下一段，多预取只是抢带宽）——这说明「更激进的预取」不等于「更快」，**预取本质上是在「带宽 × 命中率」和「浪费带宽」之间做交换**；
- 注意这里预取为何有效：`agent_scan` 是**顺序扫读**，下一段的位置几乎确定；换成 `agent_random` 同样的预取器就会大量白搬——**预取器的价值完全取决于访问模式的可预测性**（这就是第 13 讲专家预取的同一结论）；
- **E5 压缩不是「精度实验」**：本节只算字节账——FP8 让每 token 从 320 KB 掉到 160 KB，同样的 256 MB 预算能多装一倍页（48 → 97 → 195 页），停顿也从 ~832 ms 降到 ~406 ms、~193 ms（命中率 3.1% → 5.5% → 10.2%）；真实系统里还要付「反量化算力」与「精度损失」两笔账（07 讲量化、§6 的 AscMLA）；
- 还有一个隐藏收益：**压缩后页变小，同样带宽下能预取更多页**——这就是 AscMLA 在长上下文 MLA 解码上用 FP4 的动机之一。


In [ ]:
print('五个实验一页速查（同一套 KV 账本与停顿模型）')
print(f'{"实验":<14}{"关键结论":<44}{"数字":<28}')
print('-' * 86)
e1_lo = table[(1024, 8)].bytes_moved() / 1e6
e1_hi = table[(1024, 64)].bytes_moved() / 1e6
print(f'{"E1 页大小":<14}{"页越大越费带宽；容量跨过工作集才算命中":<44}'
      f'{e1_lo:.0f} → {e1_hi:.0f} MB（HBM=1024）')
print(f'{"E2 上下文":<14}{"容量固定时上下文越长命中率越低":<44}'
      f'{h2[0]:.2f} → {h2[1]:.2f} → {h2[2]:.2f}（4K→64K）')
print(f'{"E3 前缀复用":<14}{"共享前缀直接抬命中率":<44}'
      f'{h3[0]:.1%} → {h3[-1]:.1%}（0 → 75% 共享）')
print(f'{"E4 预取":<14}{"预测准就能把同步搬运变重叠":<44}'
      f'停顿 {res4[0][1].stalled_us()/1e3:.0f} → {res4[1][1].stalled_us()/1e3:.0f} ms')
print(f'{"E5 压缩":<14}{"同样的字节预算装更多 token":<44}'
      f'{st5[0]:.0f} → {st5[-1]:.0f} ms（BF16→INT4）')

# 断言：E2 的命中率必须严格递减（防止以后改参数时结论失效）
assert h2[0] > h2[-1] and st5[0] > st5[-1]


In [ ]:
# Amdahl 视角：停顿占比 = stall / (compute + stall)
COMPUTE_MS = 30.0        # 假设一个 decode step 的计算时间（示意值）

print(f'假设每步计算 {COMPUTE_MS:.0f} ms，缺页停顿占比：')
fracs = []
for ctx, m in zip(ctxs, res2):
    stall = m.stalled_us() / 1e3
    frac = stall / (COMPUTE_MS + stall)
    fracs.append(frac)
    print(f'  ctx={ctx:>6}：停顿 {stall:7.1f} ms → 停顿占比 {frac:6.1%}，'
          f'等效吞吐 {1/(COMPUTE_MS+stall)*1e3:7.3f} step/s')

assert fracs[0] < fracs[1] < fracs[2], fracs
assert fracs[2] > 0.9                      # 长上下文时停顿成为主导项
print('\n结论：上下文越长，"算得快"越没用——KV 搬运成了 Amdahl 的串行部分，必须先解决它。')

fig, ax = plt.subplots(figsize=(7.2, 3.6))
ax.bar([f'{c//1024}K' for c in ctxs], [f * 100 for f in fracs],
       color=['tab:green', 'tab:orange', 'tab:red'], alpha=0.85)
for i, f in enumerate(fracs):
    ax.text(i, f * 100, f'{f:.1%}', ha='center', va='bottom', fontsize=9)
ax.set_ylabel('停顿占总时间比例（%）'); ax.set_ylim(0, 105)
ax.set_title(f'Amdahl：计算固定 {COMPUTE_MS:.0f} ms/step 时，KV 搬运的占比')
ax.grid(alpha=0.3, axis='y')
plt.tight_layout(); plt.show()


In [ ]:
# 三种访问模式的「页热度矩阵」：横轴 = 页号，纵轴 = 时间步，颜色 = 是否访问
modes = [('decode', dict(ctx=4096, steps=96, window=512)),
         ('dialogue', dict(ctx=4096, turns=4, shared_frac=0.5, n_req=2)),
         ('agent_random', dict(ctx=4096, window=512, n_access=96, seed=3))]
P = 16
n_pages = 4096 // P

fig, axes = plt.subplots(1, 3, figsize=(13, 3.6))
for ax, (mode, kw) in zip(axes, modes):
    seq = simulate_access(mode, **kw)
    mat = np.zeros((len(seq), n_pages))
    for t, (req, pos) in enumerate(seq):
        for p in {x // P for x in pos}:
            if p < n_pages:
                mat[t, p] = 1
    ax.imshow(mat, aspect='auto', cmap='Greys', interpolation='nearest')
    ax.set_title(f'{mode}（{len(seq)} 步）', fontsize=10)
    ax.set_xlabel('页号'); ax.set_ylabel('时间步')
plt.suptitle('访问模式决定一切：竖条=顺序扫读，对角带=滑窗解码，散点=Agent 长尾回查', fontsize=11)
plt.tight_layout(); plt.show()

# 断言：三种模式的可预测性（相邻步页重合度）差异显著
def step_overlap(mode, **kw):
    seq = simulate_access(mode, **kw)
    sets = [set(x // P for x in pos) for _, pos in seq]
    vals = [len(a & b) / max(1, len(a | b)) for a, b in zip(sets, sets[1:])]
    return float(np.mean(vals))

ov_decode = step_overlap('decode', ctx=2048, steps=48, window=512)
ov_scan = step_overlap('agent_scan', ctx=2048, chunk=512, turns=2)
ov_rand = step_overlap('agent_random', ctx=2048, window=512, n_access=48, seed=3)
print(f'相邻步页重合度（IoU）：decode={ov_decode:.3f}  agent_scan={ov_scan:.3f}  agent_random={ov_rand:.3f}')
assert ov_decode > 0.9 > ov_rand, (ov_decode, ov_rand)      # 滑窗：相邻步高度重合
assert ov_scan <= 0.05, ov_scan                            # 顺序扫读：相邻步几乎零重合
assert ov_rand < 0.8, ov_rand                              # 随机回查：中等重合但没有规律


**看图说话（这三个图是本节最值得记住的一张图）：**

- **decode**：一条**对角带**——每步只碰最近窗口，页热度高度集中，LRU 天生适配；
- **dialogue**：**整段竖条重复出现**——每轮把前缀整段重读，天然适合前缀缓存；
- **agent_random**：**散点**——页热度处处为零星尖峰，LRU 刚装上就可能被下一批随机页挤掉，这就是 §6 里 Aggron / TideCache 要解决的问题场景。

定量版本（相邻步页重合度 IoU）：**decode ≈ 1.00 > agent_random ≈ 0.40 > agent_scan ≈ 0.00**。

注意这个排序容易被误读：**scan 的重合度是 0（每页一遍只读一次），但它的「可预测性」最高**（下一段位置几乎确定），所以 E4 里预取能把停顿砍 6×；random 的重合度更高，却因为不可预测而难以预取。**「重用」和「可预测」是两个不同维度**——前者决定 LRU 的上限，后者决定预取的上限，这也是第 13 讲专家预取里 `precision@k` 与本节的 `IoU` 是同源指标的原因。


## 6️⃣ 前沿对照（2026-11）

> 口径说明：以下几条只做**机制 + 代价**级别的定性对照，**不引用具体加速倍数**（不同论文的基线、硬件、负载差异太大，直接横向比倍数会误导）；引用方式为「名称 + 年份 + 一句话定位」。

| 工作（2026） | 管什么 | 关键机制 | 与本节哪块对应 |
|---|---|---|---|
| **TripleKV**（2026） | 延迟 / 命中率 | 「Predict, Prefetch, Protect」：把预测式预取搬到 KV 页，并给关键页加保护（pin），防止被 LRU 踢掉 | 本节的 **E4 前瞻预取** + 「pin 防止抖动」；第 13 讲专家预取的同构问题 |
| **Twindex**（2026） | 容量 / 带宽 | 长上下文稀疏注意力的**索引缓存 offload**：把稀疏索引（不是完整 KV）下沉到低速层，按需取回 | 本节的**分层存储**，但 offload 对象从 KV 页换成「索引元数据」——字节小两个数量级 |
| **Aggron**（2026，[arXiv:2611.26435](https://arxiv.org/abs/2611.26435)） | 延迟 / 拥塞 | 请求队列**头阻塞（HOL）**时激进压缩 agentic KV cache，把带宽让给关键路径 | 本节的**压缩（E5）** + 调度层：压缩不再只是容量优化，而是**拥塞控制手段** |
| **TideCache**（2026，[arXiv:2611.26619](https://arxiv.org/abs/2611.26619)） | 容量 / 带宽 / 复用 | 混合 LLM 服务的透明缓存管理 + **layer-wise streaming**：按层流式加载，配合异构缓存池 | 本节的**分层 + 流式搬运**；说明「按层」是天然的流水线切分单位 |
| **AscMLA**（2026） | 存储精度 / 带宽 | 长上下文 MLA 解码的 **FP4** 量化：把 latent 缓存压到 4 bit，靠反量化算力换带宽 | 本节的 **E5 压缩** + §2 的 MLA 口径（latent 维度小，压强度的收益最高） |
| **MoBA 加速**（2026） | 访问模式本身 | Block Attention 的**动态路由** + **KV-centric 执行**：让每个 query 只碰少量 block（访问模式变稀疏），而不是把所有 KV 都拉进来 | 本节的**访问模式生成器**：MoBA 等于把 `agent_random` 的全量窗口访问改造成「路由到少数块」 |
| **RoCoKV**（2026） | 容量 / 精度 | 不确定性 / 冗余感知的 KV 压缩：按 token 的重要性决定压多少 | 本节的 **E5** 的「朴素压缩」升级版：压缩粒度从「整页」细化到「按重要性」 |
| **XM-Sparse**（2026） | 容量 / 拓扑 | 跨 scale-up 内存池的稀疏注意力服务：KV 不再绑在单卡，而是放在可远程访问的内存池里 | 本节的**两层存储**扩成「机架级内存池」；带宽瓶颈从 PCIe 变成 scale-up 网络 |

**四条从对照表里能直接拿去面试的洞察：**

1. **分层是共识，压缩是变量**：所有工作都在「HBM → DRAM → 远端内存」这条轴上做文章，区别在于**压缩什么**（KV latent / 索引 / 按重要性）和**什么时候压**（常态 vs 拥塞时）；
2. **「按层」是最自然的流水线单位**：TideCache 的 layer-wise streaming 与第 12 讲的双缓冲同构——第 $l$ 层算的时候搬第 $l+1$ 层的页；
3. **预取的关键是「可预测性」而非「更聪明的模型」**：TripleKV 的 Predict 与第 13 讲的三种预取器是同一件事，顺序扫读场景一个「上一段 + 1」的朴素预测器就够用；
4. **Agent 负载把 KV 管理从「容量问题」变成了「调度问题」**：Aggron 的动机不是装不下，而是**关键请求被非关键请求的 KV 搬运堵住**（HOL blocking）。


### 为什么 Agent 场景必须重新设计 KV 管理

把本节三种访问模式放在一起看，差异一目了然：

| 维度 | 长解码（chat） | 多轮对话 | Agent 工具调用 |
|---|---|---|---|
| 访问范围 | 最近 window + sink | 每轮整段前缀 | 整段扫读 / 随机回查 |
| 时间局部性 | 极强（窗口滑动） | 强（轮间复用） | 弱（长尾随机） |
| 可预测性 | 高 | 中 | 低（除非有轨迹先验） |
| LRU 表现 | 好 | 好（配合前缀缓存） | 差（抖动） |
| 有效手段 | 页大小调优 | 前缀缓存 / 共享页 | 压缩 + 分层 + 调度隔离 |

这正是 Aggron / TideCache 这类工作出现的原因：**同样的 LRU + 同样的页大小，在 chat 负载上表现良好，在 agent 负载上会退化成抖动**。工程上的应对是三条：**压缩（少搬）、分层（搬得动）、隔离（别堵关键路径）**。


## 7️⃣ 六个常见误区（面试踩了就扣分）

| 误区 | 为什么错 | 正确说法 |
|---|---|---|
| 「KV 就是缓存，LRU 肯定最优」 | 顺序扫读 + 容量小于工作集时，LRU 会 100% 抖动；最优淘汰需要未来信息（Belady），工程上靠**预取/分层**绕过 | 「淘汰算法决定下限，访问模式与预取决定上限」 |
| 「页越大越省元数据」 | 页内碎片平均浪费 $P/2$ 个 token，且一次缺页搬更多字节（E1） | 「页大小是元数据开销与带宽浪费的权衡，甜点 16~64」 |
| 「命中率高 = 快」 | 命中率不等于搬运字节；反过来，E2 里命中率一路下滑，停顿一路飙到 800 ms | 「按 命中率 → 搬运字节 → 停顿 → 吞吐 四层汇报」 |
| 「KV 压缩一定掉精度，所以不能压」 | 压缩分对象：按重要性压、按 latent 压、按精度压，损失差异极大（RoCoKV / AscMLA） | 「先问压什么、什么时候压，再讨论损失」 |
| 「预取越多越好」 | 预取错一次就白搬一页，浪费率会吃掉收益（E4） | 「预取按浪费率设上限，并给 pin 保护」 |
| 「MLA 让 KV 消失」 | MLA 省的是缓存字节，代价是推理时必须做 up-projection（算力换显存） | 「MLA 是 latent 缓存，不是零成本」 |


## 🎯 面试考点

**Q1. KV 分页与专家分页有什么异同？**
> 同：都用「逻辑页 → 物理槽位」的页表抽象，都要处理缺页搬运与淘汰。异：KV 页按请求分配、追加写、随序列增长；专家页按 `(层, 专家)` 分配、跨请求共享、只读但随热度抖动。KV 分页的目标是**碎片与容量**，专家分页的目标是**搬运停顿**。

**Q2. GQA 的 KV 账本怎么算？为什么 128K 上下文必须靠它？**
> `2·L·n_kv·d_h·bytes`。以 L=80、n_kv=8、d_h=128、BF16 算，每 token ≈ 320 KB；128K × batch8 就是 343 GB，远超单卡。GQA 把 KV 头从 64 砍到 8，正是让长上下文可行的前提之一。

**Q3. MLA 为什么能再砍一个数量级？口径上有什么坑？**
> 它不存完整 K/V，只存低秩 latent（`d_c+d_r`，如 512+64）× L，每 token ≈ 69 KB，约为 GQA 的 21%。坑有两个：一是**单位与精度**（FP16/BF16 差 2×）；二是「省显存」的代价是推理时要多做 up-projection——**算力换显存**，不是免费。

**Q4. 前缀缓存（prefix caching）为什么收益大？什么时候没用？**
> 同一段 system prompt / 长文档被多个请求共享，页可以直接复用，命中率与首 token 延迟同时改善。没用的场景：请求互不相关（如开放域聊天）、前缀被截断或重排、或者 KV 被压缩后复用粒度对不上。

**Q5. KV offload 的带宽瓶颈怎么估算？**
> 每 token 字节 × 页大小 × 缺页次数 ÷ 有效带宽。本例 25 GB/s PCIe 下，320 KB/token × 16 token = 5 MB/页 ≈ 0.2 ms/次；如果每步缺 10 页，解码就被拖慢一倍以上——所以 offload 必须配合预取（重叠）或用更快的互连（NVLink / scale-up 内存池）。

**Q6. 预测式 KV 预取的命中率和浪费率怎么权衡？**
> 预取器越激进（前瞻越远、预测越多），命中率越高、停顿越少，但浪费带宽越多；工程上一般按「浪费率 < 10~20%」设上限，并给预取页设**保护位（pin）**防止被 LRU 立刻踢掉（TripleKV 的 Protect）。本节 E4 实测：前瞻 1 段 ~5× 加速、浪费 ~3%。

**Q7. KV 压缩会不会掉精度？**
> 会，但取决于压什么：按 token 重要性压（RoCoKV）、按 latent 压（AscMLA 的 FP4）、或直接丢尾部 token 重算，精度损失差异很大。正确回答是「分场景」：短上下文/高精度任务尽量不压；长上下文/agent 负载里压缩换来的容量收益通常大于精度损失。

**Q8. 为什么 Agent 场景必须重新设计 KV 管理？**
> 访问模式变了：chat 是滑窗（局部性强），agent 是整段扫读 + 长尾回查（局部性弱、可预测性差），LRU 会抖动；而且 agent 有「关键路径请求」与「后台请求」之分，KV 搬运造成的头阻塞会直接拖垮交互延迟（Aggron 的动机）。

**Q9. 页大小怎么选？**
> 小页省带宽、元数据开销大；大页省元数据、过度取数多。经验甜点 16~64 tokens/页；进一步优化是**自适应页大小**（热点用小页、扫描段用大页），或按层/按请求类型区分。

**Q10. 如果让你设计一个「KV 分层 + 预取」系统，指标怎么定？**
> 四层递进：命中率 → 搬运字节 → 停顿（含 overlap）→ 端到端吞吐/尾延迟；再补两个护栏：**预取浪费率**与**pin 命中率**（关键页被踢的比例）。只报命中率一定会被追问吞吐。

**Q11. KV 页表和操作系统的页表有什么可类比、有什么不能类比？**
> 可类比：页表、缺页、TLB（这里对应 slot 映射缓存）、LRU 近似（OS 用 clock，这里用 OrderedDict）。不能类比：KV 页是**追加写且只读旧页**（不需要脏页回写）、生命周期绑请求（不是进程）、且**页内容大小固定**（不需要换入换出整个地址空间）。

**Q12. 分层 KV 和 MoE 专家卸载可以共用一套基础设施吗？**
> 可以共用「页表 + 搬运引擎 + 预取器」，但调度目标不同：KV 缺页影响的是**当前请求的延迟**，专家缺页影响的是**所有并发请求的吞吐**；因此优先级、批处理策略、pin 策略都要分开设计。


## 📖 术语与参数速查

| 术语 | 含义 | 典型取值 |
|---|---|---|
| page / block | KV 的固定分配单位 | 16 tokens（vLLM 默认） |
| page table / block table | 逻辑页 → 物理槽位映射 | GPU 上的 int32 数组 |
| slot | HBM 中一块固定大小的 KV 缓存区 | 每 slot = 一页 |
| prefix caching | 跨请求复用相同前缀的页 | 命中率提升常在 10~40 个点 |
| offload | 把冷页下沉到 DRAM/SSD | 容量换带宽 |
| prefetch | 投机提前装入 | 收益取决于可预测性 |
| pin | 锁定关键页不被淘汰 | TripleKV 的 Protect |
| COW | 写时复制，共享页只在写时分裂 | 并行采样 / beam search |
| HOL blocking | 头阻塞：慢请求堵住后续请求 | Aggron 的动机 |
| layer-wise streaming | 按层流式搬运，与计算重叠 | TideCache |

**常用硬件口径（估算用，别背死）**

- PCIe 4.0 x16 有效带宽 ≈ **25 GB/s**（理论 32，实际打折）；PCIe 5.0 ≈ 50 GB/s；
- NVLink / scale-up 互连 ≈ **数百 GB/s ~ TB/s 量级**；
- HBM3e 单卡容量 80 / 141 / 192 GB 级；DRAM 容量 TB 级、带宽 100~300 GB/s；
- NVMe SSD 顺序读 3~7 GB/s、延迟几十~上百 μs。

> 面试报数时给「量级 + 假设」比给精确值更安全：**"按 PCIe 25 GB/s 估，一页 5 MB 就是 0.2 ms"**——把假设讲出来，面试官会认可你的建模习惯。


## 🛠 从本节到生产系统：改造清单

如果你明天就要在一个推理引擎里落地「KV 分层 + 预取」，按这个顺序做（每步都有明确验收指标）：

1. **先量化账本**：用真实模型的 $L, n_{kv}, d_h$ 和目标上下文算出 KV GB，确认瓶颈是容量还是带宽（本节 §2）；
2. **再分页**：block_size 16 起步，确认页表在 GPU 上、kernel 能直接索引；
3. **开前缀缓存**：先吃「system prompt / 文档复用」这波最容易的收益，验收 `prefix cache hit rate`；
4. **加分层**：HBM → DRAM offload，验收「命中率 / 搬运字节 / 停顿」三件套，别只看命中率；
5. **最后加预取**：从最朴素的「上一段 + 1」开始，设浪费率上限（10~20%），给关键页加 pin；
6. **上线后再调**：按负载类型（chat / agent / batch）分别配页大小与预取激进程度——**一套参数打不了所有负载**（§6 的核心结论）。

验收红线（可作为面试回答的模板）：**尾延迟 P99 不退化、预取浪费率 < 10%、前缀缓存命中率 > 30%（有共享前缀的场景）**。


## ✅ 自测清单 + 衔接

**手算能力（不看代码完成）**

- [ ] 给定 $L=80, n_{kv}=8, d_h=128$、BF16，算每 token KV 字节；再算 128K × batch4 的显存；
- [ ] 给定页大小 32、PCIe 20 GB/s，算一次缺页的停顿（ms）；
- [ ] 命中率 0.8 → 0.9，搬运字节一定下降吗？（答：不一定，要看每页字节与访问集合）

**动手能力**

- [ ] 不看代码写出 `KVTierManager` 的 `_evict_one`，并说明为什么淘汰时必须删 `page_slot`（否则 double-booking）；
- [ ] 把 E4 的预取器换成「预测上一段」（而非下一段），观察浪费率如何变化，解释原因；
- [ ] 给 `KVTierManager` 加一个 `pin(page_id)` 接口，实现「关键页永不淘汰」，并用 E4 轨迹验证停顿下降；
- [ ] 用 E5 的参数算：INT4 下 128K 上下文的 KV 需要多少 GB，能否塞进单卡 80 GB（batch=1）。

**概念自检**

- [ ] KV 分页 vs 专家分页的三个本质差别（生命周期 / 读写语义 / 目标函数）；
- [ ] 为什么「只看命中率」会骗人？举一个命中率高但吞吐下降的例子；
- [ ] 为什么 Agent 负载会让 LRU 抖动？给出两种缓解手段；
- [ ] 前缀缓存、预测预取、压缩、分层这四件事，分别解决「容量 / 带宽 / 延迟」中的哪一个？

---

**衔接**

- 前置：**05-KV Cache与推理内存.ipynb**（KV 字节公式、PagedAttention 的连续性论证）、**02-混合精度与显存账本.ipynb**（BF16/FP8 字节数怎么来的）、**06-推理引擎vLLM连续批处理与投机解码.ipynb**（连续批处理下的 KV 分配）；
- 同模块：**11-专家分页与显存池化.ipynb**（同一页表抽象、不同生命周期）、**12-专家卸载与CPU-GPU协同.ipynb**（双缓冲与 overlap，本节 E4 的停顿模型就是它的简化版）、**13-预测式专家预取.ipynb**（预取器设计、precision@k / 浪费率）；
- 跨模块：**04-LLM大模型/教学/19-KV缓存.ipynb**（KV 的算法视角：为什么需要、怎么压缩）；
- 一句话记住：**KV 分层不是「把缓存挪到更慢的盘」，而是「用可预测性换停顿」——预测得准，PCIe 就是免费的；预测不准，命中也救不了吞吐。**
